## In Class Activity April 21st

In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
import category_encoders as ce
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

In [ ]:
METRIC = "balanced_accuracy"
METRIC_FUNC = balanced_accuracy_score

## Load & Prep Data

In [ ]:
# load data
df = pd.read_csv("../data/adult.csv")

# convert "?" -> NaN
df = df.replace("?", np.nan)

# convert target variable to binary
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# convert gender to 0/1 (doesn't need categorical encoding since it's binary)
if "gender" in df.columns:
    df["gender"] = df["gender"].apply(lambda x: 1 if x == "Male" else 0)

df.head()

In [ ]:
# fixed train/test split for the entire notebook
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    stratify=df["income"],
    random_state=42
)

# for cross validation - maintain the same splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

# Base Modeling

## Model 1 - `RandomForest`

- no scaling required
- must encode categorical variables

### Preprocessing

In [ ]:
# how many unique values are there in each categorical
df.select_dtypes(include='str').nunique().sort_values(ascending=False)

In [ ]:
# encode categorical variables
df_encoded = df.copy()

# dummify low cardinality (<10) cats
df_encoded = pd.get_dummies(
    df_encoded,
    columns=["marital-status", "relationship", "race", "workclass"],
    drop_first=True,
    dtype=int
)

# target encode high cardinality (>=10) variables
occupation_encoder = ce.TargetEncoder(cols=["occupation"])
df_encoded["occupation_encoded"] = occupation_encoder.fit_transform(
    X=df_encoded[["occupation"]],
    y=df_encoded["income"]
)["occupation"]

native_country_encoder = ce.TargetEncoder(cols=["native-country"])
df_encoded["native_county_encoded"] = native_country_encoder.fit_transform(
    X=df_encoded[["native-country"]],
    y=df_encoded["income"]
)["native-country"]

df_encoded.head()

In [ ]:
# get training & testing data
X = df_encoded.drop(columns=[
    "income",
    "occupation",
    "fnlwgt",
    "education", # we have a numeric proxy for this
    "native-country"
])

y = df_encoded["income"]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

### Training

In [ ]:
# declare model spec
rf = RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1)

# cv on training data before fitting
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {rf_cv_scores.mean():.4f} +/- {rf_cv_scores.std():.4f}")

# fit
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_metric_score = METRIC_FUNC(y_test, rf_preds)
print(f"{METRIC}: {rf_metric_score:.4f}")

## Model 2 -  SVM

- scaling required
- must encode categorical features

### Preprocessing

- we can use `df_encoded`, but we need to scale the data as well

In [ ]:
# get training & testing data
X = df_encoded.drop(columns=[
    "income",
    "occupation",
    "fnlwgt",
    "education", # we have a numeric proxy for this
    "native-country"
])

y = df_encoded["income"]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

### Training

In [ ]:
# declare model spec
ct = ColumnTransformer([("std", StandardScaler(), make_column_selector(dtype_include="number"))])
svm = Pipeline(
    [
        ("ct", ct),
        ("svm", SVC(random_state=42, kernel="rbf")) # nonlinear kernel
    ]
)

# cv on training data before fitting
svm_cv_scores = cross_val_score(svm, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {svm_cv_scores.mean():.4f} +/- {svm_cv_scores.std():.4f}")

# fit
svm.fit(X_train, y_train)
svm_preds = svm.predict(X_test)

svm_metric_score = METRIC_FUNC(y_test, svm_preds)
print(f"{METRIC}: {svm_metric_score:.4f}")

## Model 3 - LightGBM

- tree-based: no scaling
- can handle categorical features automatically


### Preprocessing

In [ ]:
df_lgbm = df.copy()

cat_cols = df_lgbm.select_dtypes("str").columns
df_lgbm[cat_cols] = df_lgbm[cat_cols].astype("category")

df_lgbm.head()

In [ ]:
# get training & testing data
# for some reason leaving both education vars gives a slight boost to performance
X = df_lgbm.drop(columns=[
    "income",
    "fnlwgt",
])

y = df_lgbm["income"]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

### Training

In [ ]:
# declare model spec
lgbm = LGBMClassifier(random_state=42, n_jobs=-1, class_weight="balanced", verbosity=0)

# cv on training data before fitting
lgbm_cv_scores = cross_val_score(lgbm, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {lgbm_cv_scores.mean():.4f} +/- {lgbm_cv_scores.std():.4f}")

# fit
lgbm.fit(X_train, y_train)
lgbm_preds = lgbm.predict(X_test)

lgbm_metric_score = METRIC_FUNC(y_test, lgbm_preds)
print(f"{METRIC}: {lgbm_metric_score:.4f}")

## Overall Base Model Evaluations

In [ ]:
base_results = pd.DataFrame({
    "mean_cv_score": [rf_cv_scores.mean(), svm_cv_scores.mean(), lgbm_cv_scores.mean()],
    "std_cv_score": [rf_cv_scores.std(), svm_cv_scores.std(), lgbm_cv_scores.std()],
    "balanced_accuracy": [rf_metric_score, svm_metric_score, lgbm_metric_score]
}, index=["RandomForest", "SVM", "LightGBM"]).round(4).sort_values("balanced_accuracy", ascending=False)

base_results

### Inspect Errors

- gray bars show relative volume of that attribute.

- really interesting that LightGBM seems to have the highest error in most categories and at most ranges for numeric features...and yet it outperforms the other models by a significant margin.
- likewise, the svm seems to have the lowest errors across those same fields and it performed horribly
- ps: this might be because "error" here is defined as `accuracy` while we evaluated models on `balanced_accuracy`

In [ ]:
df_err = X_test.copy()
df_err["y_true"] = y_test
df_err["error_rf"] = (rf_preds != y_test).astype(int)
df_err["error_svm"] = (svm_preds != y_test).astype(int)
df_err["error_lgbm"] = (lgbm_preds != y_test).astype(int)


plt.rcParams["font.size"] = 8

fig, axes = plt.subplots(figsize=(16, 24), ncols=2, nrows=3)
axes = axes.flatten()

# age
bins_age = pd.qcut(df_err["age"], q=6)
grouped_age = df_err.groupby(bins_age)
errors_age = grouped_age[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_age = grouped_age.size()

errors_age.plot(kind="line", ax=axes[0], marker="o")

ax2 = axes[0].twinx()
ax2.bar(range(len(counts_age)), counts_age, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_age.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])

# education years
grouped_edu = df_err.groupby("educational-num")
errors_edu = grouped_edu[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_edu = grouped_edu.size()

errors_edu.plot(kind="line", ax=axes[1], marker="o")

ax2 = axes[1].twinx()
ax2.bar(range(len(counts_edu)), counts_edu, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_edu.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])

# occupation years
grouped_occ = df_err.groupby("occupation")
errors_occ = grouped_occ[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_occ = grouped_occ.size()

errors_occ.plot(kind="bar", ax=axes[2])

ax2 = axes[2].twinx()
ax2.bar(range(len(counts_occ)), counts_occ, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_occ.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])

# relationship years
grouped_relation = df_err.groupby("relationship")
errors_relation = grouped_relation[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_relation = grouped_relation.size()

errors_relation.plot(kind="bar", ax=axes[3])

ax2 = axes[3].twinx()
ax2.bar(range(len(counts_relation)), counts_relation, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_relation.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])

# race
grouped_race = df_err.groupby("race")
errors_race = grouped_race[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_race = grouped_race.size()

errors_race.plot(kind="bar", ax=axes[4])

ax2 = axes[4].twinx()
ax2.bar(range(len(counts_race)), counts_race, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_race.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])

# gender
grouped_gender = df_err.groupby("gender")
errors_gender = grouped_gender[["error_rf", "error_svm", "error_lgbm"]].mean()
counts_gender = grouped_gender.size()

errors_gender.plot(kind="bar", ax=axes[5])

ax2 = axes[5].twinx()
ax2.bar(range(len(counts_gender)), counts_gender, alpha=0.2, color="gray", zorder=0)
ax2.set_ylim(0, counts_gender.max() * 3)  # squash bars into lower portion so they don't fight the lines
ax2.set_yticks([])


plt.show()

# Create New Features

In [ ]:
df_eng = df.copy()

# missing ness
df_eng = df_eng.assign(
    missing_workclass=(pd.isna(df["workclass"])).astype(int),
    missing_occupation=(pd.isna(df["occupation"])).astype(int)
)

df_eng["net_capital"] = df_eng["capital-gain"] - df_eng["capital-loss"]
df_eng["age_binned"] = pd.qcut(df_eng["age"], q=10).astype("category")
df_eng["is_overtime_worker"] = (df_eng["hours-per-week"] > 40).astype(int)

# Evaluate New Features

## Model 1 - RandomForest

### Preprocessing

In [ ]:
# encode categorical variables
df_eng_encoded = df_eng.copy()
# dummify low cardinality (<10) cats
df_eng_encoded = pd.get_dummies(
    df_eng_encoded,
    columns=["marital-status", "relationship", "race", "workclass", "age_binned"],
    drop_first=True,
    dtype=int
)

# target encode high cardinality (>=10) variables
occupation_encoder = ce.TargetEncoder(cols=["occupation"])
df_eng_encoded["occupation_encoded"] = occupation_encoder.fit_transform(
    X=df_eng_encoded[["occupation"]],
    y=df_eng_encoded["income"]
)["occupation"]

native_country_encoder = ce.TargetEncoder(cols=["native-country"])
df_eng_encoded["native_county_encoded"] = native_country_encoder.fit_transform(
    X=df_eng_encoded[["native-country"]],
    y=df_eng_encoded["income"]
)["native-country"]

In [ ]:
# get training & testing data
X = df_eng_encoded.drop(columns=[
    "income",
    "age",
    "occupation",
    "fnlwgt",
    "capital-gain",
    "capital-loss",
    "education", # we have a numeric proxy for this
    "native-country"
])

y = df_eng_encoded["income"]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

X_train.head()

### Training

In [ ]:
# declare model spec
rf_v2 = RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1)

# cv on training data before fitting
rf_v2_cv_scores = cross_val_score(rf_v2, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {rf_v2_cv_scores.mean():.4f} +/- {rf_v2_cv_scores.std():.4f}")

# fit
rf_v2.fit(X_train, y_train)
rf_v2_preds = rf_v2.predict(X_test)

rf_v2_metric_score = METRIC_FUNC(y_test, rf_v2_preds)
print(f"{METRIC}: {rf_v2_metric_score:.4f}")

## Model 2 - SVM

### Preprocessing

In [ ]:
# na

### Training

In [ ]:
# declare model spec
ct = ColumnTransformer([("std", StandardScaler(), make_column_selector(dtype_include="number"))])
svm_v2 = Pipeline(
    [
        ("ct", ct),
        ("svm_v2", SVC(random_state=42, kernel="rbf")) # nonlinear kernel
    ]
)

# cv on training data before fitting
svm_v2_cv_scores = cross_val_score(svm_v2, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {svm_v2_cv_scores.mean():.4f} +/- {svm_v2_cv_scores.std():.4f}")

# fit
svm_v2.fit(X_train, y_train)
svm_v2_preds = svm_v2.predict(X_test)

svm_v2_metric_score = METRIC_FUNC(y_test, svm_v2_preds)
print(f"{METRIC}: {svm_metric_score:.4f}")

## Model 3 - LightGBM

### Preprocessing

In [ ]:
# get training & testing data
X = df_eng.drop(columns=[
    "income",
    "fnlwgt",
    "capital-gain",
    "capital-loss",
    "age_binned",
    "education", # we have a numeric proxy for this
])

cat_cols = X.select_dtypes("str").columns
X[cat_cols] = X[cat_cols].astype("category")

y = df_eng["income"]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

X_train.head()

### Training

In [ ]:
# declare model spec
lgbm_v2 = LGBMClassifier(random_state=42, class_weight="balanced", verbosity=0)

# cv on training data before fitting
lgbm_v2_cv_scores = cross_val_score(lgbm_v2, X_train, y_train, cv=skf, scoring=METRIC, n_jobs=-1)
print(f"cv scores: {lgbm_v2_cv_scores.mean():.4f} +/- {lgbm_v2_cv_scores.std():.4f}")

# fit
lgbm_v2.fit(X_train, y_train)
lgbm_v2_preds = lgbm_v2.predict(X_test)

lgbm_v2_metric_score = METRIC_FUNC(y_test, lgbm_v2_preds)
print(f"{METRIC}: {lgbm_v2_metric_score:.4f}")

## New Feature Evaluations

In [ ]:
eng_results = pd.DataFrame({
    "mean_cv_score": [rf_v2_cv_scores.mean(), svm_v2_cv_scores.mean(), lgbm_v2_cv_scores.mean()],
    "std_cv_score": [rf_v2_cv_scores.std(), svm_v2_cv_scores.std(), lgbm_v2_cv_scores.std()],
    "balanced_accuracy": [rf_v2_metric_score, svm_v2_metric_score, lgbm_v2_metric_score]
}, index=["RandomForest", "SVM", "LightGBM"]).round(4).sort_values("balanced_accuracy", ascending=False)

eng_results

In [ ]:
# diffs
eng_results - base_results

# Finetuning

## Tune RandomForest

In [ ]:
# get training & testing data
X = df_eng_encoded.drop(columns=[
    "income",
    "age",
    "occupation",
    "fnlwgt",
    "capital-gain",
    "capital-loss",
    "education", # we have a numeric proxy for this
    "native-country"
])

y = df_eng_encoded["income"]

X_train_rf = X.loc[train_idx]
X_test_rf = X.loc[test_idx]
y_train_rf = y.loc[train_idx]
y_test = y.loc[test_idx]

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 5, 500)
    max_depth = trial.suggest_int('max_depth', 1, 100)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    rf_optuna = RandomForestClassifier(
                    random_state=42,
                    class_weight="balanced",
                    n_jobs=-1,
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    max_features=max_features,
                    min_samples_leaf=min_samples_leaf
                )

    cv_scores = cross_val_score(rf_optuna, X_train_rf, y_train_rf, cv=skf, scoring='balanced_accuracy')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

In [ ]:
best_rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
    **study.best_params
)

best_rf.fit(X_train_rf, y_train_rf)
best_rf_preds = best_rf.predict_proba(X_test_rf)

In [ ]:
pd.DataFrame({
    "feature": X_train_rf.columns,
    "importance": best_rf.feature_importances_
}).sort_values("importance", ascending=False)

## Tune SVM

In [ ]:
# this will take eons and im almost certain it will not be good. we are skipping this.

## Tune LightGBM

In [ ]:
# get training & testing data
X = df_eng.drop(columns=[
    "income",
    "fnlwgt",
    "capital-gain",
    "capital-loss",
    "age_binned",
    "education", # we have a numeric proxy for this
])

cat_cols = X.select_dtypes("str").columns
X[cat_cols] = X[cat_cols].astype("category")

y = df_eng["income"]

X_train_gbm = X.loc[train_idx]
X_test_gbm = X.loc[test_idx]
y_train_gbm = y.loc[train_idx]
y_test = y.loc[test_idx]

In [ ]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 5, 500)
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 0, 10)
    max_depth = trial.suggest_int('max_depth', -1, 100)
    learning_rate = trial.suggest_float('learning_rate', 0.3, 0.5)
    alpha = trial.suggest_float("alpha", 0, 3)

    lgbm_optuna = LGBMClassifier(
                    random_state=42,
                    verbosity=0,
                    n_estimators=n_estimators,
                    scale_pos_weight=scale_pos_weight,
                    max_depth=max_depth,
                    learning_rate=learning_rate,
                    alpha=alpha,
                )

    cv_scores = cross_val_score(lgbm_optuna, X_train_gbm, y_train_gbm, cv=skf, scoring='balanced_accuracy')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
best_lgbm = LGBMClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
    **study.best_params
)

best_lgbm.fit(X_train_gbm, y_train_gbm)
best_lgbm_preds = best_lgbm.predict_proba(X_test_gbm)

In [ ]:
pd.DataFrame({
    "feature": X_train_gbm.columns,
    "importance": best_lgbm.feature_importances_
}).sort_values("importance", ascending=False)

# Combine Final Models

score goes down?

In [ ]:
def objective(trial):
    rf_weight = trial.suggest_float("rf_weight", 0.0, 1.0)
    lgbm_weight = 1.0 - rf_weight

    combined_probs = rf_weight * rf_preds + lgbm_weight * lgbm_preds
    y_pred_combined = (combined_probs >= 0.5).astype(int)

    return balanced_accuracy_score(y_test, y_pred_combined)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# best weights
best_rf_weight = study.best_params["rf_weight"]
best_lgbm_weight = 1.0 - best_rf_weight
print(f"Best RF Weight: {best_rf_weight:.4f}, Best LGBM Weight: {best_lgbm_weight:.4f}")


# Final Evaluation

- LGBM was the clear winner from the start. Always outperformed the other models by a wide margin.
- With tuning and some feature engineering, the RF model improved a good amount! (0.79 -> 0.83)
- Ensemble did not improve performance. In fact it went down.
- The key features were `martial_status`,  `occupation`, `net_capital`
- The LightGBM really liked my engineered `net_capital` feature and hated the others.
- The RF liked the `net_capital` and `is_overtime_worker` and hated the others.

Moving forward, I really liked the error inspection process. One thing I didn't try but I think could be really useful is using clusters as a feature. In general, I prefer to only use features that are rooted in the domain of the dataset. While I have done random transformations on numeric features (and had performance increases), the results were negligible.